<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 15 — Fine-Tuning on `credit_card_qa.csv` (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 15**.

It uses your uploaded file **`credit_card_qa.csv`** and creates **two fine-tuned models**:

1. **Model 1 — DistilBERT classifier**  
   Input: `complaint`  
   Output: `policy_category`

2. **Model 2 — DistilGPT2 generator**  
   Input: `complaint`  
   Output: `resolution`

## Why two different model types?
The two targets are **not the same kind of problem**:

- `policy_category` is a standard **classification** task
- `resolution` is much closer to a **text generation** task

This notebook therefore uses:
- **DistilBERT** for the classification task
- **DistilGPT2** for the resolution-generation task

That fits the data much better than trying to force both targets into the same model type.

## What this notebook does
- loads `credit_card_qa.csv`
- checks the dataset
- takes a **60-record fine-tuning sample**
- fine-tunes **DistilBERT** on `complaint -> policy_category`
- fine-tunes **DistilGPT2** on `complaint -> resolution`
- evaluates both models on **all records**
- also reports a stricter **holdout evaluation** on the remaining 20 records
- compares results against simple baselines
- writes a short narrative discussing the insights from fine-tuning

## Colab recommendation
Use **Runtime → Change runtime type → T4 GPU**

## Important note
The assignment brief appears to say `credit_card_aa.csv`, but your uploaded file is `credit_card_qa.csv`.  
This notebook uses the uploaded CSV directly.


## Step 0 — Install the required packages

Run this cell first.


### Important Colab note

This notebook uses a **Colab-safe install cell**.  
It keeps **pandas** and **numpy** within versions that match Colab's preinstalled environment, which avoids dependency conflicts.


In [ ]:
# Colab-safe install cell
# Avoid upgrading pandas/numpy to versions that conflict with Colab itself.
!pip -q install   "pandas==2.2.2"   "numpy>=1.26,<2.1"   "transformers>=4.44,<5"   "datasets>=2.19,<4"   "evaluate>=0.4,<1"   "accelerate>=0.30,<2"   "sentence-transformers>=3,<4"   "scikit-learn>=1.4,<2"   "rouge_score>=0.1.2,<1"

print("✅ Colab-safe packages installed.")
print("If this is the first time you ran the install cell, restart the runtime once before continuing.")

## Step 1 — Import the libraries


In [ ]:
import os
import re
import json
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
import evaluate

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)

from sentence_transformers import SentenceTransformer

set_seed(42)
random.seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUTPUT_DIR = Path("task15_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Step 2 — Upload `credit_card_qa.csv`

Upload the CSV when prompted.


In [ ]:
from google.colab import files

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file uploaded. Please upload credit_card_qa.csv")

uploaded_name = list(uploaded.keys())[0]
csv_path = f"/content/{uploaded_name}"

print("✅ Uploaded file:", csv_path)

## Step 3 — Load the dataset


In [ ]:
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:", list(df.columns))

## Step 4 — Basic cleaning

We keep only the columns needed for this task:
- `complaint`
- `policy_category`
- `resolution`


In [ ]:
required_cols = ["complaint", "policy_category", "resolution"]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].copy()

for col in required_cols:
    df[col] = df[col].astype(str).str.strip()

df = df.replace({"": np.nan}).dropna().reset_index(drop=True)
df["original_row_id"] = np.arange(len(df))

print("✅ Cleaned dataset shape:", df.shape)
display(df.head())

## Step 5 — Inspect the label structure

This is very important because it tells us what kind of modeling approach makes sense.


In [ ]:
print("Unique policy_category labels:", df["policy_category"].nunique())
print("Unique resolution texts      :", df["resolution"].nunique())

print("\nTop policy categories:")
display(df["policy_category"].value_counts().head(15).to_frame("count"))

print("\nTop resolutions:")
display(df["resolution"].value_counts().head(10).to_frame("count"))

## Step 6 — Create the 60-record fine-tuning sample

The assignment asks for a sample of **60 records** to fine-tune on.

To make the classification task more stable, we use a **coverage-aware sample**:
- first take one row from as many `policy_category` classes as possible
- then fill the rest randomly until we reach 60 rows

This same 60-row sample is used for both models.


In [ ]:
TRAIN_SAMPLE_SIZE = 60
RANDOM_STATE = 42

if TRAIN_SAMPLE_SIZE > len(df):
    raise ValueError(f"TRAIN_SAMPLE_SIZE={TRAIN_SAMPLE_SIZE} is larger than the dataset size ({len(df)}).")

rng = np.random.default_rng(RANDOM_STATE)

# 1) Take one row from each policy_category if possible
coverage_indices = []
for label, group in df.groupby("policy_category"):
    chosen_idx = group.sample(n=1, random_state=RANDOM_STATE).index[0]
    coverage_indices.append(chosen_idx)

coverage_indices = list(dict.fromkeys(coverage_indices))

# 2) Fill the remaining slots randomly
remaining_needed = TRAIN_SAMPLE_SIZE - len(coverage_indices)
remaining_pool = df.index.difference(coverage_indices)

if remaining_needed > 0:
    extra_indices = df.loc[remaining_pool].sample(n=remaining_needed, random_state=RANDOM_STATE).index.tolist()
else:
    extra_indices = []

train_indices = sorted(coverage_indices + extra_indices)
train_df = df.loc[train_indices].reset_index(drop=True)

holdout_df = df.drop(index=train_indices).reset_index(drop=True)

print("Train sample size :", len(train_df))
print("Holdout size      :", len(holdout_df))
print("Policy categories covered in train:", train_df["policy_category"].nunique(), "out of", df["policy_category"].nunique())

display(train_df.head())

## Step 7 — Save the train sample and holdout split

This is useful for transparency and reproducibility.


In [ ]:
train_df.to_csv(OUTPUT_DIR / "train_sample_60.csv", index=False)
holdout_df.to_csv(OUTPUT_DIR / "holdout_20.csv", index=False)

print("✅ Saved:")
print("-", OUTPUT_DIR / "train_sample_60.csv")
print("-", OUTPUT_DIR / "holdout_20.csv")

# Part A — Fine-tune DistilBERT for `complaint -> policy_category`


## Step 8 — Label encode `policy_category`

We encode labels using **all labels in the full dataset** so evaluation on all rows stays consistent.


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df["policy_category"])

train_df_cls = train_df.copy()
all_df_cls = df.copy()
holdout_df_cls = holdout_df.copy()

train_df_cls["label"] = label_encoder.transform(train_df_cls["policy_category"])
all_df_cls["label"] = label_encoder.transform(all_df_cls["policy_category"])
holdout_df_cls["label"] = label_encoder.transform(holdout_df_cls["policy_category"])

id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("Number of classes:", len(label_encoder.classes_))
print("First few labels:", list(label_encoder.classes_[:10]))

## Step 9 — Build Hugging Face datasets for the classifier


In [ ]:
train_ds_cls = Dataset.from_pandas(train_df_cls[["complaint", "label"]], preserve_index=False)
all_ds_cls = Dataset.from_pandas(all_df_cls[["complaint", "label"]], preserve_index=False)
holdout_ds_cls = Dataset.from_pandas(holdout_df_cls[["complaint", "label"]], preserve_index=False)

print(train_ds_cls)
print(all_ds_cls)
print(holdout_ds_cls)

## Step 10 — Load DistilBERT tokenizer and model


In [ ]:
CLS_MODEL_NAME = "distilbert-base-uncased"

cls_tokenizer = AutoTokenizer.from_pretrained(CLS_MODEL_NAME)
cls_model = AutoModelForSequenceClassification.from_pretrained(
    CLS_MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    id2label=id2label,
    label2id=label2id
)

print("✅ DistilBERT classifier loaded.")

## Step 11 — Tokenize the complaint text for classification


In [ ]:
MAX_LEN_CLS = 256

def tokenize_cls(batch):
    return cls_tokenizer(
        batch["complaint"],
        truncation=True,
        padding=False,
        max_length=MAX_LEN_CLS
    )

train_tok_cls = train_ds_cls.map(tokenize_cls, batched=True)
all_tok_cls = all_ds_cls.map(tokenize_cls, batched=True)
holdout_tok_cls = holdout_ds_cls.map(tokenize_cls, batched=True)

data_collator_cls = DataCollatorWithPadding(tokenizer=cls_tokenizer)
print("✅ Classification datasets tokenized.")

## Step 12 — Define classification metrics


In [ ]:
def compute_cls_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0)
    }

## Step 13 — Fine-tune DistilBERT

These settings are chosen to be **Colab-friendly** for a small dataset.


In [ ]:
cls_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "distilbert_policy_category"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=5,
    report_to=[],
    fp16=torch.cuda.is_available()
)

cls_trainer = Trainer(
    model=cls_model,
    args=cls_args,
    train_dataset=train_tok_cls,
    eval_dataset=holdout_tok_cls if len(holdout_tok_cls) > 0 else train_tok_cls,
    tokenizer=cls_tokenizer,
    data_collator=data_collator_cls,
    compute_metrics=compute_cls_metrics
)

cls_train_result = cls_trainer.train()
print("✅ DistilBERT fine-tuning complete.")

## Step 14 — Evaluate the classifier on all records and on the holdout set


In [ ]:
all_metrics_cls = cls_trainer.evaluate(all_tok_cls)
holdout_metrics_cls = cls_trainer.evaluate(holdout_tok_cls) if len(holdout_tok_cls) > 0 else {}

print("All-record metrics:")
print(all_metrics_cls)

print("\nHoldout metrics:")
print(holdout_metrics_cls)

## Step 15 — Generate `policy_category` predictions for every record


In [ ]:
all_preds_output = cls_trainer.predict(all_tok_cls)
all_pred_ids = np.argmax(all_preds_output.predictions, axis=-1)
all_pred_labels = label_encoder.inverse_transform(all_pred_ids)

policy_results_df = df.copy()
policy_results_df["predicted_policy_category"] = all_pred_labels
policy_results_df["policy_category_correct"] = (
    policy_results_df["policy_category"] == policy_results_df["predicted_policy_category"]
)

display(policy_results_df.head(10))

## Step 16 — Compare DistilBERT against a simple majority-class baseline


In [ ]:
majority_label = train_df["policy_category"].mode().iloc[0]
majority_preds_all = np.array([majority_label] * len(df))
majority_preds_holdout = np.array([majority_label] * len(holdout_df))

policy_baseline_metrics = {
    "majority_label": majority_label,
    "all_accuracy": accuracy_score(df["policy_category"], majority_preds_all),
    "all_macro_f1": f1_score(df["policy_category"], majority_preds_all, average="macro", zero_division=0),
    "all_weighted_f1": f1_score(df["policy_category"], majority_preds_all, average="weighted", zero_division=0),
}

if len(holdout_df) > 0:
    policy_baseline_metrics["holdout_accuracy"] = accuracy_score(holdout_df["policy_category"], majority_preds_holdout)
    policy_baseline_metrics["holdout_macro_f1"] = f1_score(holdout_df["policy_category"], majority_preds_holdout, average="macro", zero_division=0)
    policy_baseline_metrics["holdout_weighted_f1"] = f1_score(holdout_df["policy_category"], majority_preds_holdout, average="weighted", zero_division=0)

policy_baseline_metrics

# Part B — Fine-tune DistilGPT2 for `complaint -> resolution`


## Step 17 — Prepare prompt/response format for DistilGPT2

We train DistilGPT2 to continue from this prompt:

```text
Complaint: ...
Resolution:
```

and generate the target resolution text.


In [ ]:
GPT_MODEL_NAME = "distilgpt2"

gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_NAME)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

def build_prompt(complaint):
    return f"Complaint: {complaint}\nResolution:"

def build_training_text(complaint, resolution):
    return f"Complaint: {complaint}\nResolution: {resolution}{gpt_tokenizer.eos_token}"

train_df_gen = train_df.copy()
all_df_gen = df.copy()
holdout_df_gen = holdout_df.copy()

train_df_gen["train_text"] = train_df_gen.apply(lambda r: build_training_text(r["complaint"], r["resolution"]), axis=1)
all_df_gen["prompt_text"] = all_df_gen["complaint"].apply(build_prompt)
holdout_df_gen["prompt_text"] = holdout_df_gen["complaint"].apply(build_prompt)

display(train_df_gen[["complaint", "resolution", "train_text"]].head(2))

## Step 18 — Build the GPT training dataset


In [ ]:
train_ds_gen = Dataset.from_pandas(train_df_gen[["train_text"]], preserve_index=False)
print(train_ds_gen)

## Step 19 — Tokenize the GPT training texts


In [ ]:
MAX_LEN_GPT = 256

def tokenize_gen(batch):
    return gpt_tokenizer(
        batch["train_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_GPT
    )

train_tok_gen = train_ds_gen.map(tokenize_gen, batched=True, remove_columns=["train_text"])

data_collator_gen = DataCollatorForLanguageModeling(
    tokenizer=gpt_tokenizer,
    mlm=False
)

print("✅ GPT training data tokenized.")

## Step 20 — Load DistilGPT2


In [ ]:
gpt_model = AutoModelForCausalLM.from_pretrained(GPT_MODEL_NAME)
gpt_model.config.pad_token_id = gpt_tokenizer.pad_token_id

print("✅ DistilGPT2 loaded.")

## Step 21 — Fine-tune DistilGPT2 on complaint → resolution examples

Because the training set is tiny, we keep the run small and Colab-friendly.


In [ ]:
gpt_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "distilgpt2_resolution"),
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=6,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy="epoch",
    report_to=[],
    fp16=torch.cuda.is_available()
)

gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    train_dataset=train_tok_gen,
    tokenizer=gpt_tokenizer,
    data_collator=data_collator_gen
)

gpt_train_result = gpt_trainer.train()
print("✅ DistilGPT2 fine-tuning complete.")

## Step 22 — Generate `resolution` predictions for every complaint


In [ ]:
def generate_resolution(model, tokenizer, complaint, max_new_tokens=96):
    prompt = build_prompt(complaint)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN_GPT).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=3,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if "Resolution:" in decoded:
        generated = decoded.split("Resolution:", 1)[1].strip()
    else:
        generated = decoded.replace(prompt, "").strip()

    generated = re.sub(r"\s+", " ", generated).strip()
    return generated

all_generated_resolutions = []
for complaint in all_df_gen["complaint"]:
    all_generated_resolutions.append(
        generate_resolution(gpt_model, gpt_tokenizer, complaint)
    )

resolution_results_df = df.copy()
resolution_results_df["predicted_resolution"] = all_generated_resolutions

display(resolution_results_df.head(10))

## Step 23 — Evaluate the generated resolutions

For generation, exact match is usually too strict, so we report:
- **Exact Match**
- **ROUGE-L**
- **Sentence-embedding cosine similarity**

We compute metrics on:
- **all 80 records**
- the stricter **holdout 20**


In [ ]:
!pip -q install "rouge_score>=0.1.2,<1"

rouge_metric = evaluate.load("rouge")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def compute_generation_metrics(reference_texts, predicted_texts):
    exact_match = np.mean([
        str(r).strip().lower() == str(p).strip().lower()
        for r, p in zip(reference_texts, predicted_texts)
    ])

    rouge = rouge_metric.compute(predictions=predicted_texts, references=reference_texts)

    ref_emb = embedder.encode(reference_texts, convert_to_numpy=True, show_progress_bar=False)
    pred_emb = embedder.encode(predicted_texts, convert_to_numpy=True, show_progress_bar=False)

    sims = []
    for i in range(len(reference_texts)):
        sims.append(cosine_similarity(ref_emb[i:i+1], pred_emb[i:i+1])[0][0])

    return {
        "exact_match": float(exact_match),
        "rouge1": float(rouge["rouge1"]),
        "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]),
        "mean_embedding_cosine": float(np.mean(sims))
    }

In [ ]:
resolution_metrics_all = compute_generation_metrics(
    reference_texts=resolution_results_df["resolution"].tolist(),
    predicted_texts=resolution_results_df["predicted_resolution"].tolist()
)

if len(holdout_df_gen) > 0:
    # Match holdout rows back to the original full-data predictions using original_row_id
    holdout_pred_df = (
        resolution_results_df
        .set_index("original_row_id")
        .loc[holdout_df_gen["original_row_id"].tolist()]
        .reset_index()
    )

    holdout_predicted_resolutions = holdout_pred_df["predicted_resolution"].tolist()

    resolution_metrics_holdout = compute_generation_metrics(
        reference_texts=holdout_df_gen["resolution"].tolist(),
        predicted_texts=holdout_predicted_resolutions
    )
else:
    resolution_metrics_holdout = {}

print("All-record generation metrics:")
print(resolution_metrics_all)

print("\nHoldout generation metrics:")
print(resolution_metrics_holdout)

## Step 24 — Compare DistilGPT2 against a simple baseline

The baseline always outputs the most common resolution from the 60-row fine-tuning sample.


In [ ]:
majority_resolution = train_df["resolution"].mode().iloc[0]
baseline_resolution_preds_all = [majority_resolution] * len(df)

resolution_baseline_all = compute_generation_metrics(
    reference_texts=df["resolution"].tolist(),
    predicted_texts=baseline_resolution_preds_all
)

if len(holdout_df) > 0:
    baseline_resolution_preds_holdout = [majority_resolution] * len(holdout_df)
    resolution_baseline_holdout = compute_generation_metrics(
        reference_texts=holdout_df["resolution"].tolist(),
        predicted_texts=baseline_resolution_preds_holdout
    )
else:
    resolution_baseline_holdout = {}

print("Baseline resolution metrics on all records:")
print(resolution_baseline_all)

print("\nBaseline resolution metrics on holdout:")
print(resolution_baseline_holdout)

## Step 25 — Show a few prediction examples


In [ ]:
print("===== Policy Category Prediction Examples =====")
display(policy_results_df[["complaint", "policy_category", "predicted_policy_category", "policy_category_correct"]].head(10))

print("\n===== Resolution Generation Examples =====")
display(resolution_results_df[["complaint", "resolution", "predicted_resolution"]].head(10))

## Step 26 — Build a compact comparison table

This is the easiest table to screenshot or discuss in your submission.


In [ ]:
comparison_table = pd.DataFrame([
    {
        "task": "complaint -> policy_category",
        "model": "DistilBERT fine-tuned",
        "main_metric_all": all_metrics_cls.get("eval_macro_f1"),
        "secondary_metric_all": all_metrics_cls.get("eval_accuracy"),
        "main_metric_holdout": holdout_metrics_cls.get("eval_macro_f1") if len(holdout_metrics_cls) > 0 else None,
        "secondary_metric_holdout": holdout_metrics_cls.get("eval_accuracy") if len(holdout_metrics_cls) > 0 else None,
        "notes": "Classification task"
    },
    {
        "task": "complaint -> policy_category",
        "model": "Majority baseline",
        "main_metric_all": policy_baseline_metrics.get("all_macro_f1"),
        "secondary_metric_all": policy_baseline_metrics.get("all_accuracy"),
        "main_metric_holdout": policy_baseline_metrics.get("holdout_macro_f1"),
        "secondary_metric_holdout": policy_baseline_metrics.get("holdout_accuracy"),
        "notes": "Predicts most common category only"
    },
    {
        "task": "complaint -> resolution",
        "model": "DistilGPT2 fine-tuned",
        "main_metric_all": resolution_metrics_all.get("rougeL"),
        "secondary_metric_all": resolution_metrics_all.get("mean_embedding_cosine"),
        "main_metric_holdout": resolution_metrics_holdout.get("rougeL") if len(resolution_metrics_holdout) > 0 else None,
        "secondary_metric_holdout": resolution_metrics_holdout.get("mean_embedding_cosine") if len(resolution_metrics_holdout) > 0 else None,
        "notes": "Generation task"
    },
    {
        "task": "complaint -> resolution",
        "model": "Most-common-resolution baseline",
        "main_metric_all": resolution_baseline_all.get("rougeL"),
        "secondary_metric_all": resolution_baseline_all.get("mean_embedding_cosine"),
        "main_metric_holdout": resolution_baseline_holdout.get("rougeL") if len(resolution_baseline_holdout) > 0 else None,
        "secondary_metric_holdout": resolution_baseline_holdout.get("mean_embedding_cosine") if len(resolution_baseline_holdout) > 0 else None,
        "notes": "Always outputs same response"
    }
])

display(comparison_table)

## Step 27 — Write a short narrative of the fine-tuning insights

This creates a short discussion paragraph for your notebook.


In [ ]:
policy_improved = (
    (all_metrics_cls.get("eval_macro_f1", 0) or 0) >
    (policy_baseline_metrics.get("all_macro_f1", 0) or 0)
)

resolution_improved = (
    (resolution_metrics_all.get("rougeL", 0) or 0) >
    (resolution_baseline_all.get("rougeL", 0) or 0)
)

narrative = f"""
Task 15 Narrative

This experiment fine-tuned two separate models on a 60-record sample from the credit-card complaints dataset.

For the policy_category task, DistilBERT was used because the target is a structured label set. The fine-tuned classifier achieved an all-record macro F1 of {all_metrics_cls.get('eval_macro_f1'):.4f} and an accuracy of {all_metrics_cls.get('eval_accuracy'):.4f}. Compared with the majority-class baseline, this suggests that fine-tuning {'improved' if policy_improved else 'did not clearly improve'} the model's ability to map complaints to policy categories. The holdout results are especially important because they show whether the model generalizes beyond the 60 examples used for fine-tuning.

For the resolution task, DistilGPT2 was used because the target behaves like free-form text rather than a small label set. The generated responses achieved a ROUGE-L score of {resolution_metrics_all.get('rougeL'):.4f} and a mean embedding cosine similarity of {resolution_metrics_all.get('mean_embedding_cosine'):.4f} on all records. Compared with the simple baseline that always outputs the most common resolution, fine-tuning {'improved' if resolution_improved else 'did not clearly improve'} the quality of generated resolutions.

Overall, the results illustrate an important lesson: fine-tuning works best when the model type matches the output structure. Classification models suit compact label spaces, while generative models are more appropriate for highly variable textual resolutions.
""".strip()

print(narrative)

with open(OUTPUT_DIR / "task15_narrative.txt", "w", encoding="utf-8") as f:
    f.write(narrative)

print("\n✅ Narrative saved to:", OUTPUT_DIR / "task15_narrative.txt")

## Step 28 — Save the predictions and metrics


In [ ]:
policy_results_df.to_csv(OUTPUT_DIR / "policy_category_predictions.csv", index=False)
resolution_results_df.to_csv(OUTPUT_DIR / "resolution_predictions.csv", index=False)
comparison_table.to_csv(OUTPUT_DIR / "model_comparison_table.csv", index=False)

metrics_summary = {
    "classification_all_metrics": all_metrics_cls,
    "classification_holdout_metrics": holdout_metrics_cls,
    "classification_baseline_metrics": policy_baseline_metrics,
    "resolution_all_metrics": resolution_metrics_all,
    "resolution_holdout_metrics": resolution_metrics_holdout,
    "resolution_baseline_all": resolution_baseline_all,
    "resolution_baseline_holdout": resolution_baseline_holdout
}

with open(OUTPUT_DIR / "task15_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

print("✅ Saved outputs:")
print("-", OUTPUT_DIR / "policy_category_predictions.csv")
print("-", OUTPUT_DIR / "resolution_predictions.csv")
print("-", OUTPUT_DIR / "model_comparison_table.csv")
print("-", OUTPUT_DIR / "task15_metrics_summary.json")

## Step 29 — Download the main output files


In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / "policy_category_predictions.csv"))
files.download(str(OUTPUT_DIR / "resolution_predictions.csv"))
files.download(str(OUTPUT_DIR / "model_comparison_table.csv"))
files.download(str(OUTPUT_DIR / "task15_metrics_summary.json"))
files.download(str(OUTPUT_DIR / "task15_narrative.txt"))

## Step 30 — Optional: zip the entire outputs folder


In [ ]:
import shutil

archive_path = shutil.make_archive("task15_outputs_archive", "zip", OUTPUT_DIR)
print("✅ Created archive:", archive_path)

from google.colab import files
files.download(archive_path)

## Troubleshooting

### If DistilGPT2 training is too slow
Reduce:
```python
MAX_LEN_GPT = 192
```
and
```python
num_train_epochs=4
```

### If Colab runs out of memory
Reduce:
```python
per_device_train_batch_size=2
```
for the DistilGPT2 section.

### If the resolution model outputs very generic text
That can happen because:
- the dataset is tiny
- many resolutions are nearly unique
- generative fine-tuning on 60 examples is difficult

That is not a bug. It is part of the insight of the task.

### Why do we also report holdout results if the assignment says to evaluate on all records?
Because evaluating on all records includes training rows and can overstate performance.  
The holdout evaluation is a more honest check of generalization.

### Why not use DistilBERT for the resolution task too?
Because `resolution` is not a small stable label set. It behaves like natural-language output, so a generation model is a better match.
